# Assured Matching Design


## Sequence Logic
- Trail (t)
    - Week (w)
        - Period (p)

- Experiment
    - Agent Parameters
        - N_s Sellers, N_b Buyers
        - m_s units per seller, m_b units per buyer
        - Sellers' cost curve parameters, Buyers' value curve parameters (can be individual level)
        - Starting P_s value (can be 0)
    - Grid Parameters
        - Grid size
        - Torus v. square (no boundary / boundary)
    - 
- Trial
    - All agents get their cost and value curves
    - These are set at the start of the Trial and are kept constant
    - Agents get random locations at which they start
        - Each agent is forced to move at the start of the trial after the distribution
    - W Weeks are contained in each trial
- Week
    - Agents go into P periods of trading (CHANGE - drop Days)
    - Agent supply and demand curves are reinitialized
    - Agents move once at the end of the week (CHANGE)
- Period
    - Agents can place 1 trade offer per period
    - Agents can accept 1 trade offer per period
    - Agent can trade at most 1 unit per period (CHANGE)
        - If their offer is accepted, they are taken out of the trading pool
        - 
    

- Toroidal Distance Calculation
    - Given two Euclidean coordinates $(x_1, y_1)$ and $(x_2, y_2)$ measured from the bottom left hand corner as $(0, +)$ with x-width $w_x$ and y-width $w_y$.
    - Get the distance between these two coordinates, get $x$ distance, as $x_d = min(|x_1-x_2|, w_x - |x_1-x_2|)$ and $y$ distance $y_d = min(|y_1-y_2|, w_y - |y_1-y_2|)$
    - Then distance $d = \sqrt{x_d^2 + y_d^2}$
- Probabilistic matching algorithm (basic)
    - Note: there is a single parameter, $\beta$, which governs the stickiness of areas (network effects)
    - Let $p \in {1, ..., M}$ span all points on the grid inhabited by at least $1$ other player.
    - Calculate distance from this point to all other inhabited points $d_p$ and find the sum $D = \sum_p d_p$. Define the minimum distance, 
    - Define the probability of landing at any particular point, assuming no other agents move, as $P_p$.
    - From the total event probability distribution, we know $1 = \sum_p P_p$.
    - Let $P_p$ be defined by a function of distance to point $p$, the distances to each knowns point, and the parameter $\beta$ such that $P_p = (\frac{1}{d_p})^\beta \frac{1}{\sum_p (\frac{1}{d_p})^\beta}$
    - Then set $\beta \geq 0$.
    - Then the $P_p \in (0, 1]$ is a well-defined probability distribution.
    - And $\frac{\partial P_p}{\partial d_p} < 0$, $\frac{\partial P_p}{\partial d_{-p}} > 0$.
    - Let $a$ be the nearest point and $z$ be the furthest point, then as $\beta \rightarrow 1$, $P_a \rightarrow 1$; as $\beta \rightarrow 0$, $P_a / P_z \rightarrow 1$.
- Population weighted-matching
    - 

# TODO: refactor to use P_l


### Payoff Function
Weekly buyer's earnings at location $l^{*}$: $v_{w, l^{*}} = [\sum_{i, p} v_{i, p} - p_{i, p}]$
### Expected value of staying
Expected buyer's weekly earnings: $\sum_{j=1}^\infty \delta^j \hat{v}_{w+j, l^*}$
- Where $\hat{v}$ is an expected payoff in this location
- Transactions based notion: $\hat{v}_{w+1} = v_w$
- Margins based notion: $\hat{v}_{w+1} = [\sum_{i, u} v_{i, u} - M_u v_{i, u}]$ where $M_u$ is your present optimal margin at the unit level.
### Expected value of leaving
Excepted buyer's outside earnings at location $l^{-*}$: $\sum_{j=1}^\infty \delta^j \hat{v}_{w+j, l^{-*}}$
- Where $\hat{v}_{w+j, l^{-*}}$ is an expected payoff at some location not $l^*$
- Transactions based notion: $\hat{v}_{w+1, l^{-*}} = \sum_{i, p} \bar{v}_{i, p} - p_{i, p}$ (based on the price and volume information coming from other nodes)
- Margins based notion: $\hat{v}_{w+1, l^{-*}} = \sum_{i, u} v_{i, u} - \dot{M}_u v_{i, u}$
    - Where $\dot{M}_u$ is your margins expectation (starts from your original random parameters)
    - Needs to have a notion of decay rate - how quickly do margins expectations collapse towards the present realizations of the margins
        - Rate of decay is defined by the $\gamma$ parameter, with "optimists" being low $\gamma$ and "pessimists" being high $\gamma$
### Movement choice function
Stay if $\sum_{j=1}^\infty \delta^j \hat{v}_{w, l^{*}} > \sum_{l\neq*}^L (1-P_z)(P_l) \sum_{j=1}^\infty \delta^j v_{w, l} + P_z\sum_{j=1}^\infty \delta^j \hat{v}_{w, l^{*}}  - \tau$
- Where $\tau$ is a cost to move (avoidable fixed cost)
- Where $P_l$ is the probability of moving to any location $l$
- And $P_z$ is belief over "getting stuck" - the likelihood of not being able to move after requesting to move because someone moved into you ($0$ in no-move equilibrium)
### Other Location Price and Volume Information
- Agents get weekly information about the average price and weekly volume at the end of the week (when making movement decision)

# Required Changes
- Agent
    - Movement decision - yes / no (no direction)
- Travel institution
    - Movement execution
    - Collect Move / Not Move
    - Calculates probability of agent i moving to any position j
    - Execute moves in sequential random agent order
        - Someone new moves to your location - your move is cancelled
            - Side effect: closer (if pop-weighted, also larger) nodes are harder to leave
- Bargaining institution
    - Random order: Collect bids/asks at each location
    - Random order: Collect accepts
        - If your bid/ask is accepted, remove you from the order queue
- Simulation runner
    - Remove day loop (replace day loop with periods loop)
    - Week, Move only at the end of the week